# H27 c_l1-free Surrogate CNF

Corrected experiment: rebuild internal dynamic mode labels from the c_l1-free dynamic-distance reference (`eta_t + path_t + pop_t[:, :7]`), retrain the trajectory surrogate with those labels, run CNF / mode-prior CNF / guided CNF, then validate generated samples and assign them back to the scalable reference families.

This notebook is intentionally separate from the previous audit notebook. The audit explained what was wrong; this one runs the corrected experiment.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('not in Colab or Drive already unavailable:', repr(exc))

from pathlib import Path
import os

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/Colab Notebooks/final_submission_h27_flow_main_20260623'),
    Path('/content/drive/MyDrive/final_submission_h27_flow_main_20260623'),
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts').exists()), Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
PREPARED = Path('outputs/h27_context_ablation_140k_cfast_prepare/prepared_flow_pilot_data.npz')
OUT_ROOT = Path('outputs/experiments/20260622_h27_c_l1_free_surrogate_cnf')
REFERENCE_ASSIGNMENTS = Path('outputs/scalable_mode_clustering_20260604/csv/dynamic_family_assignments.csv')
REFERENCE_FEATURE_CACHE_NPZ = Path('outputs/scalable_mode_clustering_20260604/npz/scalable_dynamic_reference_features_t101_eta_path_pop.npz')
REFERENCE_FEATURE_CACHE_MANIFEST = Path('outputs/scalable_mode_clustering_20260604/npz/scalable_dynamic_reference_features_t101_eta_path_pop_manifest.json')
REFERENCE_FEATURE_CACHE = REFERENCE_FEATURE_CACHE_NPZ if REFERENCE_FEATURE_CACHE_NPZ.exists() else REFERENCE_FEATURE_CACHE_MANIFEST

BASE_CONDITION = 'CFAST_ORANGE3'
METHODS = 'CNF,HTBAL_CNF_MIXPRIOR,HTBAL_CNF_GUIDED'
METHOD_LIST = [m.strip().upper() for m in METHODS.split(',') if m.strip()]
CONDITION_SETS = [f'{BASE_CONDITION}_{m}' for m in METHOD_LIST]

RUN_METADATA = True
RUN_SMOKE = True
RUN_FULL = True
RUN_VALIDATION_AND_SCALABLE_DIVERSITY = True
FORCE_FULL = False

SMOKE_EPOCHS = 2
FULL_EPOCHS = 500
BATCH_SIZE = 256
HIDDEN = 256
FLOW_LAYERS = 8
LR = 3e-4
PATIENCE = 15
N_GENERATE = 512
SEED = 20260622

PINN_EPOCHS = 180
PINN_LOSS_DYN_CE = 0.05
PINN_SCORE_MODE = 'traj_feature'
GUIDED_STEPS = 8
GUIDED_STEP_SIZE = 0.05

HTBAL_PRIOR_ALPHA = 0.30
HTBAL_PRIOR_MIN = 0.10
MODE_WEIGHT_BETA = 0.5
MODE_WEIGHT_MAX = 5.0
HIGH_TARGET_WEIGHT = 1.5
MIXPRIOR_LATENT_SEPARATION = 2.5
MIXPRIOR_LATENT_SIGMA = 0.85
MIXPRIOR_ASSIGN_WEIGHT = 0.10
MIXPRIOR_USAGE_WEIGHT = 0.05
GENERATE_STRATEGIES = 'median,mixture'

VALIDATION_TARGETS = 'ALL'
VALIDATION_N_PER_TARGET = -1
VALIDATION_SELECTION = 'head'
VALIDATION_DT = 0.5
VALIDATION_T_MAX = 50.0
VALIDATION_LAMBDA_REORG = 35.0
SAVE_TRAJECTORIES = True
INSTALL_QUTIP_IF_MISSING = True

THREAD_ENV = {
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
}

assert PREPARED.exists(), f'missing prepared artifact: {PREPARED}'
assert REFERENCE_ASSIGNMENTS.exists(), f'missing reference assignments: {REFERENCE_ASSIGNMENTS}'
assert REFERENCE_FEATURE_CACHE.exists(), (
    f'missing reference feature cache. expected either {REFERENCE_FEATURE_CACHE_NPZ} '
    f'or {REFERENCE_FEATURE_CACHE_MANIFEST}'
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('methods:', METHOD_LIST)
print('condition sets:', CONDITION_SETS)
print('prepared:', PREPARED)
print('reference feature cache:', REFERENCE_FEATURE_CACHE)
print('out:', OUT_ROOT.resolve())

In [ ]:
def run(cmd, env=None):
    cmd = list(map(str, cmd))
    print('\n$ ' + ' '.join(cmd), flush=True)
    merged_env = os.environ.copy()
    merged_env.update(THREAD_ENV)
    merged_env['PYTHONUNBUFFERED'] = '1'
    if env:
        merged_env.update(env)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged_env)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'command failed with exit code {returncode}: {cmd}')
    return subprocess.CompletedProcess(cmd, returncode)

def find_script(name):
    candidates = [Path('scripts') / name, Path.cwd() / 'scripts' / name]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path.cwd(), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        try:
            for p in root.rglob(name):
                if p.parent.name == 'scripts':
                    return p
        except Exception as exc:
            print('script search skipped:', root, repr(exc))
    raise FileNotFoundError(name)

TRAIN_SCRIPT = find_script('train_h27_c_l1_free_surrogate_cnf.py')
VALIDATE_SCRIPT = find_script('validate_h27_cfast_generated_simulator_latest.py') if (Path('scripts') / 'validate_h27_cfast_generated_simulator_latest.py').exists() else find_script('validate_h27_cfast_generated_simulator.py')
ASSIGN_SCRIPT = find_script('assign_h27_generated_to_scalable_dynamic_reference.py')
print('train:', TRAIN_SCRIPT)
print('validate:', VALIDATE_SCRIPT)
print('assign:', ASSIGN_SCRIPT)

run(['python3', '-m', 'py_compile', TRAIN_SCRIPT])
run(['python3', '-m', 'py_compile', VALIDATE_SCRIPT])
run(['python3', '-m', 'py_compile', ASSIGN_SCRIPT])

if INSTALL_QUTIP_IF_MISSING:
    try:
        import qutip  # noqa
        print('qutip ok')
    except ModuleNotFoundError:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'qutip'])

In [ ]:
def train_cmd(run_name, epochs, n_generate, extra=None):
    cmd = [
        'python3', '-u', TRAIN_SCRIPT,
        '--prepared', PREPARED,
        '--out-root', OUT_ROOT,
        '--run-name', run_name,
        '--base-condition', BASE_CONDITION,
        '--methods', METHODS,
        '--epochs', epochs,
        '--pinn-epochs', PINN_EPOCHS,
        '--pinn-loss-dyn-ce', PINN_LOSS_DYN_CE,
        '--pinn-score-mode', PINN_SCORE_MODE,
        '--batch-size', BATCH_SIZE,
        '--hidden', HIDDEN,
        '--flow-layers', FLOW_LAYERS,
        '--lr', LR,
        '--patience', PATIENCE,
        '--n-generate', n_generate,
        '--htbal-prior-alpha', HTBAL_PRIOR_ALPHA,
        '--htbal-prior-min', HTBAL_PRIOR_MIN,
        '--mode-weight-beta', MODE_WEIGHT_BETA,
        '--mode-weight-max', MODE_WEIGHT_MAX,
        '--high-target-weight', HIGH_TARGET_WEIGHT,
        '--mixprior-latent-separation', MIXPRIOR_LATENT_SEPARATION,
        '--mixprior-latent-sigma', MIXPRIOR_LATENT_SIGMA,
        '--mixprior-assign-weight', MIXPRIOR_ASSIGN_WEIGHT,
        '--mixprior-usage-weight', MIXPRIOR_USAGE_WEIGHT,
        '--guided-steps', GUIDED_STEPS,
        '--guided-step-size', GUIDED_STEP_SIZE,
        '--generate-strategies', GENERATE_STRATEGIES,
        '--c-l1-free-reference-assignments', REFERENCE_ASSIGNMENTS,
        '--c-l1-free-reference-feature-cache', REFERENCE_FEATURE_CACHE,
        '--c-l1-free-mode-column', 'dynamic_family_id',
        '--seed', SEED,
    ]
    if extra:
        cmd += extra
    return cmd

if RUN_METADATA:
    run(train_cmd('smoke', 1, 8, ['--metadata-only', '--no-progress']))

if RUN_SMOKE:
    run(train_cmd('smoke', SMOKE_EPOCHS, 16, ['--force', '--no-progress']))

if RUN_FULL:
    extra = ['--force'] if FORCE_FULL else []
    run(train_cmd('full', FULL_EPOCHS, N_GENERATE, extra))

In [ ]:
if RUN_VALIDATION_AND_SCALABLE_DIVERSITY:
    run_dir = OUT_ROOT / 'full'
    for condition_set in CONDITION_SETS:
        generated = run_dir / f'{condition_set}_generated_samples.npz'
        if not generated.exists():
            print('skip missing generated:', generated)
            continue

        val_out = run_dir / f'simulator_validation_{condition_set}'
        cmd = [
            'python3', '-u', VALIDATE_SCRIPT,
            '--generated', generated,
            '--out-dir', val_out,
            '--condition-set', condition_set,
            '--targets', VALIDATION_TARGETS,
            '--n-per-target', VALIDATION_N_PER_TARGET,
            '--selection', VALIDATION_SELECTION,
            '--lambda-reorg', VALIDATION_LAMBDA_REORG,
            '--t-max', VALIDATION_T_MAX,
            '--dt', VALIDATION_DT,
            '--print-every', 20,
        ]
        if SAVE_TRAJECTORIES:
            cmd.append('--save-trajectories')
        run(cmd)

        detail = val_out / 'csv' / f'{generated.stem}_simulator_validation_detail.csv'
        trajectories = val_out / 'npz' / f'{generated.stem}_sampled_trajectories.npz'
        assign_out = run_dir / f'scalable_reference_assignment_{condition_set}'
        run([
            'python3', '-u', ASSIGN_SCRIPT,
            '--detail', detail,
            '--trajectories', trajectories,
            '--condition-set', condition_set,
            '--run-label', f'{OUT_ROOT.name}/full',
            '--reference-assignments', REFERENCE_ASSIGNMENTS,
            '--reference-feature-cache', REFERENCE_FEATURE_CACHE,
            '--out-dir', assign_out,
            '--components', 'eta,path,pop',
            '--success-only',
            '--target-match-only',
            '--generated-chunk-size', 128,
            '--seed', SEED,
        ])
        print('validation summary:', val_out / 'csv' / f'{generated.stem}_simulator_validation_summary.csv')
        print('scalable summary:', assign_out / 'csv')

In [ ]:
import pandas as pd

run_dir = OUT_ROOT / 'full'
summary_paths = {
    'mode_guidance': OUT_ROOT / 'metadata' / 'cnf_mode_prior_mode_guidance_summary.csv',
    'branch_summary': OUT_ROOT / 'metadata' / 'cnf_mode_prior_internal_branch_summary.csv',
    'surrogate_metrics': run_dir / 'pinntraj_surrogate_split_metrics.csv',
}
for name, path in summary_paths.items():
    print('\n#', name, path, 'exists=', path.exists())
    if path.exists() and path.suffix == '.csv':
        display(pd.read_csv(path).head(20))

for condition_set in CONDITION_SETS:
    print('\n#', condition_set)
    paths = {
        'loss': run_dir / f'{condition_set}_loss_history.csv',
        'test_metrics': run_dir / f'{condition_set}_test_metrics.csv',
        'physical': run_dir / f'{condition_set}_generated_physical_summary.csv',
        'validation': run_dir / f'simulator_validation_{condition_set}' / 'csv' / f'{condition_set}_generated_samples_simulator_validation_summary.csv',
    }
    assign_dir = run_dir / f'scalable_reference_assignment_{condition_set}' / 'csv'
    if assign_dir.exists():
        summaries = sorted(assign_dir.glob('*_scalable_reference_summary.csv'))
        if summaries:
            paths['scalable_reference'] = summaries[0]
    for name, path in paths.items():
        print('\n##', name, path, 'exists=', path.exists())
        if path.exists() and path.suffix == '.csv':
            display(pd.read_csv(path).head(20))